# 04 3D Geologic Model (Spangler 2024)

**Series:** Tribal Soils and Geology

## The Western South Dakota 3D Geologic Model

Spangler (2024) published the first regional-scale volumetric 3D geologic
model of western South Dakota as a USGS data release (DOI: 10.5066/P9LK4QHJ).
The model covers all of western South Dakota from west of the Missouri River
to the Wyoming border including Pine Ridge entirely.

The model provides 25 subsurface horizon rasters representing the top
elevation of each major stratigraphic unit, plus 35 fault surfaces.
These rasters allow us to compute, for any point on the reservations:

- **Depth to any formation**: how deep is the Ogallala aquifer here?
  Where is the Pierre Shale thick vs. thin?
- **Formation thickness**: subtract adjacent horizons to produce isopach maps
- **Fault locations**: where do faults potentially compartmentalize aquifers
  or create structural complexity?

This dataset has never been visualized in a Tribal land sovereignty context.
This notebook provides that first look.

**Required data:** `data/raw/geology/WSouthDakota3D.gdb/`
Download from: https://doi.org/10.5066/P9LK4QHJ

In [ ]:
# Imports
import sys
from pathlib import Path
REPO_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "src").is_dir() and (parent / "config" / "config.yaml").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from the tribal-soils-geology repository or a subdirectory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import warnings, numpy as np, pandas as pd
import geopandas as gpd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches, contextily as ctx, yaml
from shapely.geometry import box
import rasterio
from rasterio.plot import show as rio_show
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping
import numpy as np
from rasterio.warp import reproject, Resampling

from src.constants import (
    CRS_GEOGRAPHIC, CRS_PROJECTED, CRS_WEB, REPO_ROOT as _REPO_ROOT,
    OUTPUTS_DIR, FIGURES_DIR, PINE_RIDGE_BBOX,     STUDY_BBOX, WSD_3D_MODEL, WSD_STRATIGRAPHY, WSD_KEY_UNITS,
)
from src.loaders import load_tribal_boundaries
from src.sovereignty import print_data_acknowledgment, generate_citations, attach_provenance
warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline
with open(_REPO_ROOT/"config"/"config.yaml") as f: CONFIG = yaml.safe_load(f)
TEAL="#007A6E"; TEAL_LT="#E0F4F2"; GRAY="#566573"; TERRACOTTA="#C0392B"
def despine(ax):
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
primary = load_tribal_boundaries(["Pine Ridge"])
print(f"Ready. Nations: {len(primary)}")

In [ ]:
# Print data acknowledgement at the top of every notebook
print_data_acknowledgment(source_keys=["usgs_3d_model"])

## Load Model Metadata and Boundary

In [ ]:
from src.loaders import (
    load_wsd_3d_model_boundary, load_wsd_fault_points,
    load_wsd_model_units, load_wsd_horizon_raster,
)

# Model boundary
print("Loading 3D model boundary...")
model_boundary = load_wsd_3d_model_boundary()

if model_boundary.empty:
    print()
    print("WSouthDakota3D.gdb not found.")
    print("Download from: https://doi.org/10.5066/P9LK4QHJ")
    print("  Files needed: WSouthDakota3D.gdb.zip, WSD_NonspatialTables.zip")
    print("  Extract to: data/raw/geology/")
else:
    print(f"Model boundary loaded: {len(model_boundary)} polygon(s)")
    bb = model_boundary.total_bounds
    print(f"Extent: {bb[0]:.2f}W, {bb[1]:.2f}N to {bb[2]:.2f}E, {bb[3]:.2f}N")

# Model unit descriptions
print()
print("Loading model unit descriptions...")
model_units = load_wsd_model_units()
if not model_units.empty:
    print(f"Model units table: {len(model_units)} rows, {len(model_units.columns)} columns")
    print(f"Columns: {model_units.columns.tolist()}")
    print(model_units.head(5).to_string())

# These values are defined before any raster or depth-map cells run.
gdb_path = WSD_3D_MODEL["gdb_path"]
HORIZON_NAME_MAP = {
    "Ogallala Group": "WSD_TopOgallalaGroup",
    "Fox Hills Formation": "WSD_TopFoxHillsFormation",
    "Pierre Shale": "WSD_TopPierreShale",
    "Niobrara Formation": "WSD_TopNiobraraFormation",
    "Hell Creek Formation": "WSD_TopHellCreekFormation",
    "Madison Group": "WSD_TopMadisonGroup",
    "Precambrian basement": "WSD_TopPrecambrianbasement",
}

# These values are defined before any raster or depth-map cells run.
gdb_path = WSD_3D_MODEL["gdb_path"]
HORIZON_NAME_MAP = {
    "Ogallala Group": "WSD_TopOgallalaGroup",
    "Fox Hills Formation": "WSD_TopFoxHillsFormation",
    "Pierre Shale": "WSD_TopPierreShale",
    "Niobrara Formation": "WSD_TopNiobraraFormation",
    "Hell Creek Formation": "WSD_TopHellCreekFormation",
    "Madison Group": "WSD_TopMadisonGroup",
    "Precambrian basement": "WSD_TopPrecambrianbasement",
}

In [ ]:
# Fault points
print("Loading fault surfaces...")
faults = load_wsd_fault_points()

if not faults.empty:
    print(f"Fault points loaded: {len(faults):,}")
    print(f"Columns: {faults.columns.tolist()}")
    if "Name" in faults.columns or "name" in faults.columns:
        name_col = "Name" if "Name" in faults.columns else "name"
        print(f"\nFault names:")
        print(faults[name_col].value_counts().head(15).to_string())
else:
    print("Fault data not loaded download the GDB first.")

In [ ]:
from src.constants import WSD_3D_MODEL
print("Tables path:", WSD_3D_MODEL["tables_path"])
print("Exists:", WSD_3D_MODEL["tables_path"].exists())

geology_dir = REPO_ROOT/"data"/"raw"/"geology"
print("\nAll contents of data/raw/geology/:")
for item in sorted(geology_dir.rglob("*")):
    print(f"  {item.relative_to(geology_dir)}")

## Fault Surfaces in Western South Dakota

In [ ]:
if not faults.empty:
    fig, ax = plt.subplots(figsize=(14, 10))

    if not model_boundary.empty:
        model_boundary.to_crs(CRS_WEB).plot(
            ax=ax, facecolor="none", edgecolor="#F57F17",
            linewidth=1.5, alpha=0.6, zorder=2
        )

    faults.to_crs(CRS_WEB).plot(
        ax=ax, color=TERRACOTTA, marker=".", markersize=3,
        alpha=0.5, zorder=3
    )

    try:
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, alpha=0.5)
    except Exception: pass

    ax.set_axis_off()
    ax.set_title(
        f"Fault Surfaces for Western South Dakota 3D Model\n"
        f"USGS 3D Model (Spangler 2024) | {len(faults):,} fault surface points",
        fontsize=12, fontweight="bold"
    )
    legend_el = [
        mpatches.Patch(facecolor="none", edgecolor="#F57F17", label="3D model extent"),
        plt.Line2D([0],[0], marker=".", color=TERRACOTTA, markersize=1,
                   linestyle="", label="Fault surfaces"),
    ]
    ax.legend(handles=legend_el, fontsize=9, loc="lower left", framealpha=0.9)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR/"04_fault_map.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Fault data not loaded. Download GDB to enable this visualization.")
    

In [ ]:
# Build a scientifically ordered full-extent model from all WSD_Top* rasters.
from src.geology3d import build_interactive_model, horizon_catalog

interactive_path = OUTPUTS_DIR/"04_full_3d_geology_model.html"
catalog_3d = horizon_catalog(gdb_path)
print(f"Rendering {len(catalog_3d)} model-top rasters in published geological order")
display(catalog_3d)
model_figure = build_interactive_model(
    interactive_path,
    faults=faults,
    boundaries=primary,
    stride=CONFIG["analysis"]["model_3d_stride"],
    vertical_exaggeration=CONFIG["analysis"]["vertical_exaggeration"],
)
print(f"Standalone interactive model: {interactive_path}")
model_figure.show()

In [ ]:
# Real horizon layer names discovered from subdataset listing
HORIZON_LAYERS = {
    "Ogallala Group":        "WSD_TopOgallalaGroup",
    "Fox Hills Formation":   "WSD_TopFoxHillsFormation",
    "Pierre Shale":          "WSD_TopPierreShale",
    "Niobrara Formation":    "WSD_TopNiobraraFormation",
    "Madison Group":         "WSD_TopMadisonGroup",
    "Precambrian basement":  "WSD_TopPrecambrianbasement",
}

n = len(HORIZON_LAYERS)
fig, axes = plt.subplots(2, 3, figsize=(20, 12), constrained_layout=True)
axes = axes.flatten()

for ax, (display_name, layer_name) in zip(axes, HORIZON_LAYERS.items()):
    subdataset_path = f"OpenFileGDB:{gdb_path}:{layer_name}"
    try:
        with rasterio.open(subdataset_path) as src:
            data = src.read(1, masked=True)
            rio_show(src, ax=ax, cmap="terrain", title=None)
            img = ax.get_images()[0]
            cbar = fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label("Elevation (m)", fontsize=7)
            ax.set_title(display_name, fontsize=11, fontweight="bold")
            ax.set_axis_off()
            print(f"{display_name}: {data.min():.1f} to {data.max():.1f} m, "
                  f"{src.width}x{src.height}, CRS: {src.crs}")
    except Exception as e:
        ax.text(0.5, 0.5, f"Could not load:\n{display_name}\n{e}",
                ha="center", va="center", fontsize=8, transform=ax.transAxes)
        ax.set_axis_off()
        print(f"Failed to load {display_name} ({layer_name}): {e}")

fig.suptitle("USGS 3D Geologic Model with Horizon Surfaces\n"
             "Spangler (2024), Western South Dakota",
             fontsize=14, fontweight="bold")
fig.savefig(FIGURES_DIR/"04_horizon_surfaces.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
try:
    import py3dep
except ImportError:
    py3dep = None
    print("py3dep is not installed. Recreate the conda environment from environment.yml.")

DEM_RESOLUTION_M = 90   # coarse enough to avoid timeouts/memory issues at full extent

bounds = model_boundary.total_bounds  # (minx, miny, maxx, maxy) in model_boundary's CRS
bounds_geo = model_boundary.to_crs(CRS_GEOGRAPHIC).total_bounds  # py3dep expects lon/lat

print(f"Requesting 3DEP DEM at {DEM_RESOLUTION_M}m resolution over the full model extent...")
print(f"Bounds (lon/lat): {bounds_geo}")

try:
    if py3dep is None:
        raise ImportError("py3dep is unavailable")
    dem = py3dep.get_map(
        "DEM",
        tuple(bounds_geo),
        resolution=DEM_RESOLUTION_M,
        geo_crs=CRS_GEOGRAPHIC,
        crs=CRS_PROJECTED,
    )
    print(f"DEM loaded: shape={dem.shape}, resolution={DEM_RESOLUTION_M}m")
except Exception as e:
    print(f"3DEP request failed: {e}")
    dem = None

In [ ]:
KEY_HORIZON = "Pierre Shale"
layer_name = HORIZON_NAME_MAP.get(KEY_HORIZON)

if layer_name is None:
    print(f"'{KEY_HORIZON}' not in HORIZON_NAME_MAP.")
elif dem is None:
    print("DEM not loaded run the 3DEP cell above first.")
else:
    subdataset_path = f"OpenFileGDB:{gdb_path}:{layer_name}"
    print(f"Loading horizon raster: {KEY_HORIZON} -> {layer_name}")

    with rasterio.open(subdataset_path) as horizon_ds:
        horizon_crs = horizon_ds.crs
        horizon_transform = horizon_ds.transform
        horizon_data = horizon_ds.read(1, masked=True)
        horizon_nodata = horizon_ds.nodata

    dem_arr = dem.values.astype("float64")
    dem_crs = dem.rio.crs
    dem_transform = dem.rio.transform()
    dem_shape = dem_arr.shape

    print(f"Horizon CRS: {horizon_crs}, shape: {horizon_data.shape}")
    print(f"DEM CRS: {dem_crs}, shape: {dem_shape}")

    horizon_on_dem = np.full(dem_shape, np.nan, dtype="float64")
    reproject(
        source=horizon_data.filled(np.nan) if hasattr(horizon_data, "filled") else horizon_data,
        destination=horizon_on_dem,
        src_transform=horizon_transform,
        src_crs=horizon_crs,
        dst_transform=dem_transform,
        dst_crs=dem_crs,
        resampling=Resampling.bilinear,
        src_nodata=horizon_nodata if horizon_nodata is not None else np.nan,
        dst_nodata=np.nan,
    )

    depth = dem_arr - horizon_on_dem
    depth_masked = np.ma.masked_invalid(depth)

    valid = depth_masked.compressed()
    print(f"\nDepth to {KEY_HORIZON}:")
    print(f"  Range: {valid.min():.1f} to {valid.max():.1f} m")
    print(f"  Median: {np.median(valid):.1f} m")
    print(f"  % negative (horizon above DEM; inspect datum, resolution, and model uncertainty): {(valid < 0).mean()*100:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

im = ax.imshow(
    depth_masked,
    cmap="viridis_r",
    extent=(dem_transform[2], dem_transform[2] + dem_transform[0]*dem_shape[1],
            dem_transform[5] + dem_transform[4]*dem_shape[0], dem_transform[5]),
    vmin=0,
    vmax=np.percentile(valid[valid > 0], 98),  # clip extreme tail for a readable scale
)
cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
cbar.set_label("Depth to Pierre Shale (m)", fontsize=10)

ax.set_axis_off()
ax.set_title(
    "Depth to Pierre Shale\n"
    "Surface DEM (3DEP, 90m) minus horizon elevation (Spangler 2024)",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
fig.savefig(FIGURES_DIR/"04_depth_to_pierre_shale.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Diagnose cells where the modeled Pierre top is at or above the DEM
fig, ax = plt.subplots(figsize=(14, 10))

# Negative differences are diagnostic mismatches, not an outcrop classification by themselves
nonphysical_mask = depth_masked < 0
plot_data = np.ma.masked_where(nonphysical_mask, depth_masked)

im = ax.imshow(
    plot_data, cmap="viridis_r",
    extent=(dem_transform[2], dem_transform[2] + dem_transform[0]*dem_shape[1],
            dem_transform[5] + dem_transform[4]*dem_shape[0], dem_transform[5]),
    vmin=0, vmax=np.percentile(valid[valid > 0], 98),
)
nonphysical_plot = np.ma.masked_where(~nonphysical_mask, depth_masked)
ax.imshow(
    np.ma.filled(nonphysical_plot, np.nan) * 0,  # constant value, just need the mask
    cmap="Greys", vmin=0, vmax=1,
    extent=(dem_transform[2], dem_transform[2] + dem_transform[0]*dem_shape[1],
            dem_transform[5] + dem_transform[4]*dem_shape[0], dem_transform[5]),
    alpha=np.where(nonphysical_mask.filled(False), 0.6, 0)[::1],
)

cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
cbar.set_label("Depth to Pierre Shale (m, buried areas only)", fontsize=10)

ax.set_axis_off()
ax.set_title(
    "Depth to Pierre Shale\n"
    "Gray = modeled top at/above DEM (diagnostic) | Color = positive DEM-minus-horizon difference",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
fig.savefig(FIGURES_DIR/"04_depth_to_pierre_shale_diagnostic.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
KEY_HORIZON = "Ogallala Group"
layer_name = HORIZON_NAME_MAP.get(KEY_HORIZON)

if layer_name is None:
    print(f"'{KEY_HORIZON}' not in HORIZON_NAME_MAP.")
elif dem is None:
    print("DEM not loaded: run the 3DEP cell above first.")
else:
    subdataset_path = f"OpenFileGDB:{gdb_path}:{layer_name}"
    print(f"Loading horizon raster: {KEY_HORIZON} -> {layer_name}")

    with rasterio.open(subdataset_path) as horizon_ds:
        horizon_crs = horizon_ds.crs
        horizon_transform = horizon_ds.transform
        horizon_data = horizon_ds.read(1, masked=True)
        horizon_nodata = horizon_ds.nodata

    # dem is an xarray DataArray from py3dep so pull out its grid info
    dem_arr = dem.values.astype("float64")
    dem_crs = dem.rio.crs
    dem_transform = dem.rio.transform()
    dem_shape = dem_arr.shape
    dem_nodata = dem.rio.nodata

    print(f"Horizon CRS: {horizon_crs}, shape: {horizon_data.shape}")
    print(f"DEM CRS: {dem_crs}, shape: {dem_shape}")

    # Resample the horizon raster onto the DEM's grid (reprojects + regrids in one step)
    horizon_on_dem = np.full(dem_shape, np.nan, dtype="float64")
    reproject(
        source=horizon_data.filled(np.nan) if hasattr(horizon_data, "filled") else horizon_data,
        destination=horizon_on_dem,
        src_transform=horizon_transform,
        src_crs=horizon_crs,
        dst_transform=dem_transform,
        dst_crs=dem_crs,
        resampling=Resampling.bilinear,
        src_nodata=horizon_nodata if horizon_nodata is not None else np.nan,
        dst_nodata=np.nan,
    )

    # Depth to formation = surface elevation - horizon elevation
    # Positive = formation lies below the surface (expected, normal case)
    # Negative = modeled top exceeds DEM; may reflect exposure, datum/grid mismatch, or model uncertainty
    depth = dem_arr - horizon_on_dem
    depth_masked = np.ma.masked_invalid(depth)

    valid = depth_masked.compressed()
    print(f"\nDepth to {KEY_HORIZON}:")
    print(f"  Range: {valid.min():.1f} to {valid.max():.1f} m")
    print(f"  Median: {np.median(valid):.1f} m")
    print(f"  % negative (horizon above DEM; not automatically outcrop): {(valid < 0).mean()*100:.1f}%")

## Depth-to-Formation Maps

In [ ]:
# Map user-friendly names to actual GDB horizon layer names (WSD_Top<Name> pattern)
HORIZON_NAME_MAP = {
    "Ogallala Group":       "WSD_TopOgallalaGroup",
    "Fox Hills Formation":  "WSD_TopFoxHillsFormation",
    "Pierre Shale":         "WSD_TopPierreShale",
    "Niobrara Formation":   "WSD_TopNiobraraFormation",
    "Hell Creek Formation": "WSD_TopHellCreekFormation",
    "Madison Group":        "WSD_TopMadisonGroup",
    "Precambrian basement": "WSD_TopPrecambrianbasement",
    # add remaining units from the 25-layer set as needed
}

KEY_HORIZON = "Ogallala Group"   # change to explore other formations

layer_name = HORIZON_NAME_MAP.get(KEY_HORIZON)
if layer_name is None:
    print(f"'{KEY_HORIZON}' not in HORIZON_NAME_MAP. Add its WSD_Top... name first.")
    print(f"Known friendly names: {list(HORIZON_NAME_MAP.keys())}")
else:
    subdataset_path = f"OpenFileGDB:{gdb_path}:{layer_name}"
    print(f"Loading horizon raster: {KEY_HORIZON} -> {layer_name}")
    try:
        with rasterio.open(subdataset_path) as horizon_ds:
            print(f"  CRS: {horizon_ds.crs}")
            print(f"  Shape: {horizon_ds.height} x {horizon_ds.width}")
            print(f"  Bounds: {horizon_ds.bounds}")
            horizon_data = horizon_ds.read(1, masked=True)
            horizon_profile = horizon_ds.profile
            print(f"  Elevation range: {horizon_data.min():.1f} to {horizon_data.max():.1f} m")

        # DEPTH CALCULATION needs a land-surface DEM, not yet loaded
        # depth = surface_dem - horizon_elevation
        # dem_ds = <load surface DEM here, matched to horizon_ds CRS/extent>
        # depth_data = dem_data - horizon_data
        print()
        print("Horizon loaded successfully. Depth calculation still needs a surface DEM ")
        print("no DEM source is wired up in this notebook yet.")
    except Exception as e:
        print(f"Failed to load {layer_name}: {e}")

In [ ]:
# FileGDB inventories use different GDAL APIs for vector/table and raster content.
import fiona
from src.geology3d import horizon_catalog

vector_table_layers = sorted(fiona.listlayers(str(gdb_path)))
model_tops = horizon_catalog(gdb_path)
print(f"Vector/table layers exposed by Fiona ({len(vector_table_layers)}):")
for layer in vector_table_layers:
    print(f"  {layer}")
print(f"Raster model-top subdatasets exposed by Rasterio ({len(model_tops)}):")
display(model_tops[["hierarchy_key", "name", "age", "layer", "cross_cutting"]])
print("The intrusive surface is cross-cutting; it is not a stratigraphic interval boundary.")

## Cross-Section Generator

In [ ]:
# West-east cross-section sampled from every available horizon raster.
from shapely.geometry import LineString
from src.geology3d import sample_cross_section

transect = LineString([(-103.45, 43.15), (-101.55, 43.15)])
cross_section = sample_cross_section(transect, n_points=500, gdb_path=gdb_path)
horizon_columns = [c for c in cross_section.columns
                   if c not in {"longitude", "latitude", "distance_km"}]

fig, ax = plt.subplots(figsize=(15, 8))
for column in horizon_columns:
    ax.plot(cross_section["distance_km"], cross_section[column], linewidth=1, label=column)
ax.set_xlabel("Distance along transect (km, west to east)")
ax.set_ylabel("Modeled horizon elevation (m)")
ax.set_title("Pine Ridge cross-section sampled from Spangler (2024) horizon rasters")
ax.legend(fontsize=6, ncol=3, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.grid(alpha=.2)
plt.tight_layout()
cross_section.to_csv(OUTPUTS_DIR/"04_pine_ridge_cross_section.csv", index=False)
fig.savefig(FIGURES_DIR/"04_cross_section_real.png", dpi=150, bbox_inches="tight")
plt.show()

## Model Uncertainty: A Note for Users

The Spangler (2024) model README contains critical guidance on uncertainty.
Points that geologists and resource managers should be aware of:

**Scale:** The model is regional-scale. Subsurface horizons are constrained
by well log control points that are widely spaced. Local variability in
formation depth and thickness is not captured.

**Well log density:** The density of control wells varies across the study
area. Areas with fewer wells have greater uncertainty in horizon elevations.
Pine Ridge have relatively sparse well log records compared to
surrounding areas, a monitoring equity gap that directly degrades model
accuracy on Tribal lands.

**Faults:** The 35 modeled fault surfaces represent interpreted fault traces,
not precisely located structures. Fault geometry at depth carries significant
uncertainty and is not visualized in this series.

**Use:** The model is appropriate for regional resource assessment, aquifer
characterization, and geologic framework studies. It is not a substitute for
local-scale investigation (test wells, geophysical surveys, surface mapping).

**How Tribal data helps:** Every well log drilled on Pine Ridge,
formatted to the `well_log_template.xlsx` standard, directly reduces model
uncertainty. The template is designed to produce data compatible with USGS
well log databases.

In [ ]:
# Print citations
print(generate_citations(["usgs_3d_model"]))